### Merging Infrastructure Hex Edits and Making Sector Totals

In [1]:
import geopandas as gpd
import pandas as pd
from functools import reduce

pd.set_option('display.max_columns', None)

### Merge From H3 Edits Files

In [ ]:
#hex geopackes here
#Here we have all the H3 edited files, by type of data summarized within them
files = [
    "\hex_line_infrastructure.gpkg",
    "\hex_point_infrastructure2.gpkg",
    "\hex_polygon_infrastructure.gpkg"
]

# Read each into a GeoDataFrame
gdfs = [gpd.read_file(f) for f in files]

# Merge all on 'hex_ID'
gdf_merged = reduce(
    lambda left, right: pd.merge(left, right, on="h3_ID", how="outer"),
    gdfs
)

# this will keep the geometry from the first file
gdf_merged = gpd.GeoDataFrame(gdf_merged, geometry="geometry")
#list all column headers, for sorting
column_names_list = gdf_merged.columns.tolist()
pd.Series(column_names_list).to_csv("columns.csv", index=False)

In [5]:
gdf_merged.drop(columns=["geometry_y"], inplace=True)

In [ ]:
#save where H3 edits are going
gdf_merged.to_file('C\hex_infrastructure_merged.gpkg', driver="GPKG")

In [ ]:
#read again
hex_path = "\hex_infrastructure_merged.gpkg"
#Define GPD:
gdf_merged = gpd.read_file(hex_path)

### Make Sector Total Columns

In [ ]:
#Assign Sub-Sectors to Sectors
category_map = {
    "Transportation": ["T_Mi_Index", "Amtrak Stations", "Aviation Locations", "Bus Stops",
                        "Intermodal Locations", "RO/RO Freight", "Rail Bridges", "Rail Crossings"],
    "Energy": ["E_Mi_Index", "Battery-Energy Storage Stations",
                "Electric Substations of Switching Stations", "Gas and Oil Plugged Wells", "LNG Storage",
                "Natural Gas Compressor Stations", "Petroleum Port", "Petroleum Pumping Stations", "Petroleum Terminals",
                "Power Plants", "Biodiesel Plant", "Ethanol Plant", "Ethanol Transloading", "Alternative Fueling Stations"],
    "Water & Wastewater": ["Public Drinking Water", "Sewer Point Data", "Wastewater Treatment", "Water Point Data",
                            "Water Storage Tanks"],
    "Healthcare & Public Health": ["Ambulance Services", "Assisted Living", "Dialysis Center", "Diagnostic Imaging",
                                    "Disability Facilities", "Health Education", "Health Practitioners", "Home Health Care", "Hospitals",
                                    "Medical Laboratories", "Mental Health", "Nursing Care Facilities", "Outpatient Care Centers",
                                    "Outpatient Mental Health", "Pharmacies", "Psychiatric Hospitals",
                                    "Red Cross", "Rehabilitation", "Residential Care Facilities", "Retirement Community",
                                    "Specialty Hospitals", "Public Health Departments", "Family Reunification Centers", "Family Services"],
    "Communications": ["Antenna Structure", "Broadband", "Cellular Towers", "Land Mobile Broadcast Towers", "Microwave Towers", "Paging Transmission Towers"],
    "Government Facilities": ["Courthouses", "Higher Education", "Private Schools", "Public Libraries", "Public Schools",
                                "Regional FBI Building", "School Bus Storage", "State Agency Buildings", "USPS Locations",
                                "USPS Plants", "Waste and Recycling Facilities"],
    "Emergency Services": ["Correctional Facilities", "Debris Storage", "Dry Hydrants", "Emergency Services", "Fire Stations",
                            "Homeless Shelters", "Local EOC", "Local Emergency Shelters", "Local Law Enforcement",
                            "Public Works Departments", "Religious Organizations", "State EOC", "VDEM Regional Office",
                            "VDEM State Office", "Emergency Center"],
    "Food & Agriculture": ["Food Banks", "Food Services", "Refridgerated Warehouse", "Retail Grocer", "SNAP Retailers"],
    "Financial": ["Credit Unions", "Banks", "ATM"],
    "Commercial Facilities": ["Major Event Center", "Major Sport Venues"],
    "Nuclear": ["Evacuation Assembly Center"],
    "Dams": ["Dam or Levee"]
    }

for new_col, cols in category_map.items():
    existing_cols = [c for c in cols if c in gdf_merged.columns]
    
    if existing_cols:
        gdf_merged[new_col] = (
            gdf_merged[existing_cols]
            .fillna(0)
            .sum(axis=1)
            .astype(int)
        )
    else:
        gdf_merged[new_col] = 0


gdf_merged.to_file("\hex_infrastructure_merged.gpkg", driver="GPKG")

In [ ]:
len(gdf_merged)

### Make an Infrastructure Index

In [ ]:
#making the index for risk 
#read hex bin huge layer
hex_path = "\hex_infrastructure_merged.gpkg"
#Define GPD:
gdf_merged = gpd.read_file(hex_path)

In [ ]:
summary_cols = [
    "Commercial Facilities",
    "Communications",
    "Dams",
    "Emergency Services",
    "Energy",
    "Financial",
    "Food & Agriculture",
    "Government Facilities",
    "Healthcare & Public Health",
    "Nuclear",
    "Transportation",
    "Water & Wastewater",
]

# Ensure numeric (invalid values → NaN)
gdf_merged[summary_cols] = gdf_merged[summary_cols].apply(pd.to_numeric, errors="coerce")

# Create total column (skip NaNs automatically)
gdf_merged["total"] = (
    gdf_merged[summary_cols]
    .sum(axis=1, min_count=1)   # keeps NaN if all inputs are NaN
    .fillna(0)                  # replace all-NaN rows with 0
    .round()
    .astype("Int64")            # nullable integer
)

# Min-max normalization (0–1)
min_val = gdf_merged["total"].min()
max_val = gdf_merged["total"].max()

if min_val == max_val:
    gdf_merged["Total_Index"] = 100
else:
    gdf_merged["Total_Index"] = (
        (gdf_merged["total"] - min_val) / (max_val - min_val)
    ) * 99 + 1

print(gdf_merged["Total_Index"].describe())


In [ ]:
#save and overwrite with additions
gdf_merged.to_file("\hex_infrastructure_merged.gpkg", driver="GPKG")

In [ ]:
#making a shapefile 
#reread layer
hex_path = "\hex_infrastructure_merged.gpkg"
#Define GPD:
gdf_merged = gpd.read_file(hex_path)

gdf_merged.to_file(r'\hex_infrastructure_merged.shp', driver='ESRI Shapefile')

### Make Totals by Sector CSV

In [ ]:
#read hex bin huge layer
hex_path = "\hex_infrastructure_merged.gpkg"
#Define GPD:
gdf_merged = gpd.read_file(hex_path)

In [15]:
totals = gdf_merged.select_dtypes(include="number").sum()

cols = ["Communications", "Government Facilities",
    "Healthcare & Public Health",
    "Nuclear",
    "Transportation",
    "Water & Wastewater",
    "Commercial Facilities",
    "Dams",
    "Emergency Services",
    "Energy",
    "Financial",
    "Food & Agriculture"]
totals = gdf_merged[cols].sum()

totals_df = totals.reset_index()
totals_df.columns = ["Category", "Total"]
totals_df.to_csv("selected_totals.csv", index=False)